# Water Quality Prediction: Benchmark Notebook 

## Challenge Overview

Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust **machine learning model** capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.

Participants will be provided with a dataset containing three water quality parameters — **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus** — collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.

Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.

This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.

**About the Notebook:**  

In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict **water quality parameters** using features derived from the **Landsat**, **TerraClimate**, and **NASADEM** datasets. Specifically, six spectral bands — **Blue**, **Green**, **Red**, **NIR** (Near Infrared), **SWIR16** (Shortwave Infrared 1), and **SWIR22** (Shortwave Infrared 2) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). From TerraClimate, we incorporate **PET** (Potential Evapotranspiration), **PPT** (Precipitation), **Tmax** (Maximum Temperature), **Soil** (Soil Moisture), and **AET** (Actual Evapotranspiration). From NASADEM, we incorporate topographic features: **Elevation**, **Slope**, and **Flow Accumulation**.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, Landsat, TerraClimate, and NASADEM features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, climatic, and topographic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

Please note that this notebook serves only as a starting point. Several assumptions were made during the data extraction and model development process, which you may find opportunities to improve upon. Participants are encouraged to explore additional features, enhance preprocessing techniques, or experiment with different regression algorithms to optimize predictive performance.

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

## Response Variable

Before building the model, we first load the **water quality training dataset**. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding **measured values** for the three key water quality parameters — **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

In [ ]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

## Predictor Variables

Now that we have our water quality dataset, the next step is to gather the predictor variables from the **Landsat** and **TerraClimate** datasets. In this notebook, we demonstrate how to **load previously extracted satellite and climate data** from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.

For more detailed guidance on the original data extraction process, you can review the Landsat and TerraClimate example notebooks available on the Planetary Computer portal:

- [Landsat-c2-l2 - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook)  
- [Terraclimate - Example-Notebook](https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook)

We have used selected spectral bands — **Blue**, **Green**, **Red**, **NIR** (Near Infrared), **SWIR16** (Shortwave Infrared 1), and **SWIR22** (Shortwave Infrared 2) — and computed key spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, water color, and water content characteristics that influence water quality variability.

The predictor features include:

- **Blue** – Sensitive to water color and dissolved organic matter.  
- **Green** – Useful for detecting water color and surface reflectance changes.  
- **Red** – Helps in identifying suspended sediments and chlorophyll.  
- **NIR** – Helps in identifying vegetation and suspended matter in water.  
- **SWIR16** – Provides information on surface dryness and sediment concentration.  
- **SWIR22** – Sensitive to surface moisture and turbidity variations in water bodies.  
- **NDMI** – Derived from NIR and SWIR16, indicates moisture and vegetation–water interaction.  
- **MNDWI** – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.

In addition, we incorporate **TerraClimate** variables: **PET** (Potential Evapotranspiration), **PPT** (Precipitation), **Tmax** (Maximum Temperature), **Soil** (Soil Moisture), and **AET** (Actual Evapotranspiration). These capture climatic and hydrological conditions that influence water quality.

We also incorporate topographic features from **NASADEM**: **Elevation** (height above sea level), **Slope** (terrain steepness in degrees), and **Flow Accumulation** (upstream contributing cell count via D8 routing). These capture terrain-driven drainage patterns, flow convergence, and landscape position that strongly affect water quality.

### **Tip 1**

Participants are encouraged to experiment with different combinations of **Landsat** bands or even include data from other public satellite data sources. By creating mathematical combinations of bands, you can derive various spectral indices that capture surface and environmental characteristics.

### Loading Pre-Extracted Landsat Data

In this notebook, we **load previously extracted Landsat data** from CSV files generated in a separate extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.

Participants are expected to generate their own data extraction CSV files by running the dedicated Landsat extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how individual bands and indices like **NDMI** were computed. Using these pre-extracted CSV files simplifies preprocessing and is ideal for large-scale environmental and water quality analysis.

### **Tip 2**

In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of Band 2 within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.

In [ ]:
# Load Landsat features: Latitude, Longitude, Sample Date, nir, green, swir16, swir22, NDMI, MNDWI
landsat_train_features = pd.read_csv("landsat_features_training_200.csv")
display(landsat_train_features.head(5))

In [ ]:
# If NDMI and MNDWI columns are of type object, convert them to float
landsat_train_features['NDMI'] = landsat_train_features['NDMI'].astype(float)
landsat_train_features['MNDWI'] = landsat_train_features['MNDWI'].astype(float)

In [ ]:
# Feature Engineering: Create additional spectral indices from available bands
# (nir, green, swir16, swir22 are available; blue and red are not in this dataset)

# NDWI - Normalized Difference Water Index (green-nir variant)
landsat_train_features['NDWI'] = (landsat_train_features['green'] - landsat_train_features['nir']) / \
                                  (landsat_train_features['green'] + landsat_train_features['nir'] + 1e-8)

# SWIR ratio indicates moisture and mineral content
landsat_train_features['swir_ratio'] = landsat_train_features['swir22'] / \
                                        (landsat_train_features['swir16'] + 1e-8)

# NIR/Green ratio relates to chlorophyll and suspended matter
landsat_train_features['nir_green_ratio'] = landsat_train_features['nir'] / \
                                             (landsat_train_features['green'] + 1e-8)

print(f"Added 3 new spectral features: NDWI, swir_ratio, nir_green_ratio")

### Loading Pre-Extracted TerraClimate Data

We load TerraClimate features from CSV files with columns: **Latitude**, **Longitude**, **Sample Date**, **pet** (Potential Evapotranspiration), **ppt** (Precipitation), **tmax** (Maximum Temperature), **soil** (Soil Moisture), and **aet** (Actual Evapotranspiration). These climate variables capture hydrological and meteorological conditions that influence water quality.

In [ ]:
# Load TerraClimate features: Latitude, Longitude, Sample Date, pet, ppt, tmax, soil, aet
Terraclimate_df = pd.read_csv("terraclimate_features_training.csv")
display(Terraclimate_df.head(5))

## Joining the Predictor Variables and Response Variables

Now that we have extracted our predictor variables, we need to join them with the response variables. We use the **combine_datasets** function to merge the water quality targets, Landsat features, and TerraClimate features. The **concat** function from pandas is particularly useful for this step.

In [5]:
# Combine datasets horizontally (along columns) using pandas concat function.
def combine_datasets(*datasets):
    '''
    Returns a horizontally concatenated dataset.
    Accepts any number of DataFrames (e.g. water quality targets, Landsat, TerraClimate, NASADEM).
    Duplicate columns are automatically removed.
    '''
    data = pd.concat(list(datasets), axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [6]:
# Load NASADEM features: Latitude, Longitude, Sample Date, elevation, slope, flow_accumulation
nasadem_train_df = pd.read_csv("nasadem_features_training_.csv")
display(nasadem_train_df.head(5))

# Combining water quality targets, Landsat features, TerraClimate features, and NASADEM features into a single dataset.
wq_data = combine_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, nasadem_train_df)
display(wq_data.head(5))

In [ ]:
# Feature Engineering: Extract temporal features from Sample Date
# Seasonality strongly affects water quality (rainfall patterns, temperature, evaporation)

wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'], dayfirst=True)
wq_data['month'] = wq_data['Sample Date'].dt.month
wq_data['year'] = wq_data['Sample Date'].dt.year
wq_data['day_of_year'] = wq_data['Sample Date'].dt.dayofyear

# Cyclical encoding for month - captures that December and January are adjacent
# This helps the model understand seasonal patterns without discontinuities
wq_data['month_sin'] = np.sin(2 * np.pi * wq_data['month'] / 12)
wq_data['month_cos'] = np.cos(2 * np.pi * wq_data['month'] / 12)

print(f"Added temporal features: month, year, day_of_year, month_sin, month_cos")
print(f"Date range: {wq_data['Sample Date'].min()} to {wq_data['Sample Date'].max()}")

### Handling Missing Values

Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.

In [7]:
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
wq_data.isna().sum()

In [ ]:
# Outlier Removal: Remove extreme values in target variables
# Extreme outliers can distort model training and hurt generalization

def remove_outliers(df, columns, multiplier=3):
    """Remove outliers using IQR method with configurable multiplier"""
    df_clean = df.copy()
    initial_count = len(df_clean)
    
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - multiplier * IQR
        upper = Q3 + multiplier * IQR
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
    
    removed = initial_count - len(df_clean)
    print(f"Removed {removed} outlier rows ({100*removed/initial_count:.1f}%)")
    return df_clean

target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
print(f"Rows before outlier removal: {len(wq_data)}")
wq_data = remove_outliers(wq_data, target_cols, multiplier=3)
print(f"Rows after outlier removal: {len(wq_data)}")

## Model Building

Now let us select the columns required for our model-building exercise. We will consider the Landsat spectral bands (**blue**, **green**, **red**, **nir**, **swir16**, **swir22**), derived indices (**NDMI**, **MNDWI**), TerraClimate variables (**pet**, **ppt**, **tmax**, **soil**, **aet**), NASADEM topographic features (**elevation**, **slope**, **flow_accumulation**), and additional engineered features as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.

In [ ]:
# Select all spectral bands, derived indices, temporal features, and target variables
# Using all available features provides the model with more information to learn patterns

feature_cols = [
    # Landsat bands (available in dataset)
    'green', 'nir', 'swir16', 'swir22',
    # Pre-computed spectral indices
    'NDMI', 'MNDWI',
    # Derived spectral indices
    'NDWI', 'swir_ratio', 'nir_green_ratio',
    # TerraClimate features
    'pet', 'ppt', 'tmax', 'soil', 'aet',
    # NASADEM topographic features
    'elevation', 'slope', 'flow_accumulation',
    # Temporal features for seasonality
    'month_sin', 'month_cos'
]

target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

wq_data = wq_data[feature_cols + target_cols]
print(f"Using {len(feature_cols)} features: {feature_cols}")

### **Tip 3**

We are developing individual models for each water quality parameter using a common set of features: **blue**, **green**, **red**, **nir**, **swir16**, **swir22**, **NDMI**, **MNDWI**, TerraClimate variables (**pet**, **ppt**, **tmax**, **soil**, **aet**), NASADEM topographic features (**elevation**, **slope**, **flow_accumulation**), and derived spectral indices. However, participants are encouraged to experiment with different feature combinations to build more robust machine learning models.

## Helper Functions

### Train and Test Split
We will now split the data into 70% training data and 30% test data. Scikit-learn (sklearn) is a robust library for machine learning in Python. The `model_selection` module in scikit-learn provides the `train_test_split` function, which can be used for this purpose.

### Feature Scaling
Before initiating model training, we may need to perform various data preprocessing steps. Here, we demonstrate scaling of the predictor variables (spectral bands and indices) using StandardScaler.

Feature scaling is an essential preprocessing step for numerical features. Many machine learning algorithms—such as gradient descent methods, KNN, and linear or logistic regression—require scaling to achieve optimal performance. Scikit-learn provides several scaling utilities. In this notebook, we use **StandardScaler**, which transforms the data so that each feature has a mean of 0 and a standard deviation of 1.

### Model Training
Now that we have the data in a format suitable for machine learning, we can begin training our models. In this demonstration notebook, we build three separate regression models—one for each target water quality parameter: **Total Alkalinity**, **Electrical Conductance**, and **Dissolved Reactive Phosphorus**. Each model is trained independently to capture the unique relationships between the satellite-derived features and each water quality parameter.

We use the **Random Forest Regressor** from the scikit-learn library for model training. Scikit-learn offers a wide range of regression algorithms, along with powerful parameter tuning and customization options.

For model training, the predictor variables (e.g., spectral bands, NDMI, MNDWI, and derived indices) are stored in an array `X`, and the response variable (one of the water quality parameters) is stored in an array `Y`. Note that the response variable should not be included in `X`. Also, latitude, longitude, and sample date are excluded from the predictor variables since they serve only as spatial and temporal references.

### Model Evaluation
After training the models for the three water quality parameters, the next step is to evaluate their performance. Each regression model—Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus—is assessed using:

- **R² Score**: Measures how well the model explains the variance in the observed values.  
- **RMSE (Root Mean Square Error)**: Quantifies the average magnitude of prediction errors.

Together, these metrics help determine how effectively each model captures variations in water quality across locations and sampling dates. Scikit-learn provides built-in functions to compute both metrics. Participants may also explore additional evaluation techniques or custom metrics to enhance model assessment.

### **Tip 4**

There are many data preprocessing methods available that may help improve model performance. Participants are encouraged to explore various preprocessing techniques as well as different machine learning algorithms to build a more robust model.

In [9]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train, use_grid_search=True):
    """
    Train a Random Forest model with optional hyperparameter tuning.
    Grid search finds optimal parameters using cross-validation.
    """
    if use_grid_search:
        param_grid = {
            'n_estimators': [200, 300],
            'max_depth': [8, 12, 16],
            'min_samples_split': [5, 10],
            'min_samples_leaf': [2, 4, 6],
            'max_features': ['sqrt', 'log2']
        }
        
        rf = RandomForestRegressor(random_state=42, n_jobs=-1, max_samples=0.8)
        grid_search = GridSearchCV(
            rf, param_grid, 
            cv=5,              # 5-fold cross-validation
            scoring='r2',      # Optimize for R²
            n_jobs=-1,         # Use all CPU cores
            verbose=1
        )
        grid_search.fit(X_train_scaled, y_train)
        
        print(f"Best parameters: {grid_search.best_params_}")
        print(f"Best CV R²: {grid_search.best_score_:.3f}")
        return grid_search.best_estimator_
    else:
        # Fallback to improved defaults without grid search
        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=12,
            min_samples_split=5,
            min_samples_leaf=2,
            max_samples=0.8,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train_scaled, y_train)
        return model

def evaluate_model(model, X_scaled, y_true, dataset_name="Test"):
    y_pred = model.predict(X_scaled)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n{dataset_name} Evaluation:")
    print(f"R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    return y_pred, r2, rmse

## Model Workflow (Pipeline)

The complete model development process follows a structured pipeline to ensure consistency, reproducibility, and clarity. Each stage in the workflow is modularized into independent functions that can be reused for different water quality parameters. This modular approach streamlines the process and makes the workflow easily adaptable to new datasets or parameters in the future.

The pipeline automates the sequence of steps — from data preparation to evaluation — for each target parameter. The same set of predictor variables is used, while the response variable changes for each of the three targets: *Total Alkalinity (TA)*, *Electrical Conductance (EC)*, and *Dissolved Reactive Phosphorus (DRP)*. By maintaining a consistent framework, comparisons across models remain fair and interpretable.

In [10]:
def run_pipeline(X, y, param_name="Parameter"):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    print(f"{'='*60}")
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Scale
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    
    # Train
    model = train_model(X_train_scaled, y_train)
    
    # Evaluate (in-sample)
    y_train_pred, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, "Train")
    
    # Evaluate (out-sample)
    y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")
    
    # Return summary
    results = {
        "Parameter": param_name,
        "R2_Train": r2_train,
        "RMSE_Train": rmse_train,
        "R2_Test": r2_test,
        "RMSE_Test": rmse_test
    }
    return model, scaler, pd.DataFrame([results])

### Model Training and Evaluation for Each Parameter

In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. The input feature set (`X`) remains the same across all three models, while the target variable (`y`) changes for each parameter. 

For every parameter, the `run_pipeline()` function is executed, which handles data preprocessing, model training, and both in-sample and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different water quality indicators.

In [11]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

### Model Performance Summary

After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. 

Such a summary provides a quick overview of how well each model captures the variability in each parameter and highlights any differences in predictive accuracy.

In [12]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

## Submission

Once you are satisfied with your model’s performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for a set of test locations provided in the **Submission_template.csv** file. 

The predicted results can then be uploaded to the challenge platform for evaluation.

In [ ]:
test_file = pd.read_csv("submission_template.csv")
display(test_file.head(5))

In [ ]:
# Load validation Landsat features: Latitude, Longitude, Sample Date, blue, green, red, nir, swir16, swir22, NDMI, MNDWI
landsat_val_features = pd.read_csv("landsat_features_validation.csv")
display(landsat_val_features.head(5))

In [ ]:
# Load TerraClimate validation features: Latitude, Longitude, Sample Date, pet, ppt, tmax, soil, aet
Terraclimate_val_df = pd.read_csv("terraclimate_features_validation.csv")
display(Terraclimate_val_df.head(5))

Similarly, participants can use the **Landsat** and **TerraClimate** data extraction demonstration notebooks to produce feature CSVs for their **validation** data. The Landsat validation file should contain: Latitude, Longitude, Sample Date, blue, green, red, nir, swir16, swir22, NDMI, MNDWI. The TerraClimate validation file should contain: Latitude, Longitude, Sample Date, pet, ppt, tmax, soil, aet.

Participants should save their own extracted files in the same format and column schema; doing so will allow this benchmark notebook to load the validation features directly and run smoothly.

In [16]:
# If NDMI and MNDWI columns are of type object, convert them to float
if landsat_val_features['NDMI'].dtype == object:
    landsat_val_features['NDMI'] = landsat_val_features['NDMI'].astype(float)
if landsat_val_features['MNDWI'].dtype == object:
    landsat_val_features['MNDWI'] = landsat_val_features['MNDWI'].astype(float)

# Load NASADEM validation features
nasadem_val_df = pd.read_csv("nasadem_features_validation_.csv")

# Consolidate Landsat, TerraClimate, and NASADEM features in a single dataframe
val_data = pd.DataFrame({
    'Longitude': landsat_val_features['Longitude'].values,
    'Latitude': landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    'green': landsat_val_features['green'].values,
    'nir': landsat_val_features['nir'].values,
    'swir16': landsat_val_features['swir16'].values,
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': Terraclimate_val_df['pet'].values,
    'ppt': Terraclimate_val_df['ppt'].values,
    'tmax': Terraclimate_val_df['tmax'].values,
    'soil': Terraclimate_val_df['soil'].values,
    'aet': Terraclimate_val_df['aet'].values,
    'elevation': nasadem_val_df['elevation'].values,
    'slope': nasadem_val_df['slope'].values,
    'flow_accumulation': nasadem_val_df['flow_accumulation'].values,
})

# Apply the same feature engineering as training data
val_data['NDWI'] = (val_data['green'] - val_data['nir']) / (val_data['green'] + val_data['nir'] + 1e-8)
val_data['swir_ratio'] = val_data['swir22'] / (val_data['swir16'] + 1e-8)
val_data['nir_green_ratio'] = val_data['nir'] / (val_data['green'] + 1e-8)

# Temporal features
val_data['Sample Date'] = pd.to_datetime(val_data['Sample Date'], dayfirst=True)
val_data['month'] = val_data['Sample Date'].dt.month
val_data['month_sin'] = np.sin(2 * np.pi * val_data['month'] / 12)
val_data['month_cos'] = np.cos(2 * np.pi * val_data['month'] / 12)

print(f"Validation data shape: {val_data.shape}")
print(f"Added spectral indices, TerraClimate features, NASADEM features, and temporal features to validation data")

In [17]:
# Impute the missing values
val_data = val_data.fillna(val_data.median(numeric_only=True))

In [ ]:
# Extract the same features used for training (must match exactly)
feature_cols = [
    'green', 'nir', 'swir16', 'swir22',
    'NDMI', 'MNDWI',
    'NDWI', 'swir_ratio', 'nir_green_ratio',
    'pet', 'ppt', 'tmax', 'soil', 'aet',
    'elevation', 'slope', 'flow_accumulation',
    'month_sin', 'month_cos'
]
submission_val_data = val_data[feature_cols]
print(f"Validation features: {list(submission_val_data.columns)}")
display(submission_val_data.head())

In [19]:
submission_val_data.shape

In [20]:
# --- Predicting for Total Alkalinity ---
X_sub_scaled_TA = scaler_TA.transform(submission_val_data)
pred_TA_submission = model_TA.predict(X_sub_scaled_TA)

# --- Predicting for Electrical Conductance ---
X_sub_scaled_EC = scaler_EC.transform(submission_val_data)
pred_EC_submission = model_EC.predict(X_sub_scaled_EC)

# --- Predicting for Dissolved Reactive Phosphorus ---
X_sub_scaled_DRP = scaler_DRP.transform(submission_val_data)
pred_DRP_submission = model_DRP.predict(X_sub_scaled_DRP)

In [21]:
submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA_submission,
    'Electrical Conductance': pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission
})

In [22]:
#Displaying the sample submission dataframe
display(submission_df.head())

In [23]:
#Dumping the predictions into a csv file.
submission_df.to_csv("/tmp/submission.csv",index = False)

In [ ]:
session.sql("""
    PUT file:///tmp/submission.csv
    'snow://workspace/USER$.PUBLIC."Ian-EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print("File saved! Refresh the browser to see the files in the sidebar")

### Upload submission file on platform

Upload the `submission.csv` file on the challenge platform to generate your score on the leaderboard.

## Conclusion

Now that you have learned a basic approach to model training, it’s time to explore your own techniques and ideas! Feel free to modify any of the functions presented in this notebook to experiment with alternative preprocessing steps, feature engineering strategies, or machine learning algorithms. 

We look forward to seeing your enhanced model and the insights you uncover. Best of luck with the challenge!